# ML-04 — Search Intelligence Data Contract

**Lane:** Content Performance Prediction / Refresh Opportunity Scoring

This contract uses the full FlyRank warehouse release. I use **March 2026** as the mid-panel feature month and **April 2026** as the forward outcome month. The goal is to rank content pages for review using signals that were available before the outcome window.

> The warehouse is gated. This notebook expects an `HF_TOKEN` Colab Secret or a secure token prompt. Never paste the token into a public cell.

## 0. Connect to the warehouse

The daily fact is the source for time-series features and future outcomes. DuckDB reads the Parquet data remotely so the notebook does not load the full ~79M-row table into pandas.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub scikit-learn

In [30]:
!pip install -U duckdb

In [65]:
import os
import getpass
import duckdb
from huggingface_hub import hf_hub_download

# Get Hugging Face token securely
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token: ")

# Download the March 2026 daily fact file
daily_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print("Daily file downloaded successfully:")
print(daily_file)

# Connect DuckDB
con = duckdb.connect()

# Use the downloaded local Parquet file
TABLES = {
    "daily": f"read_parquet('{daily_file}')"
}

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-31"

OUTCOME_START = "2026-04-01"
OUTCOME_END = "2026-04-30"

print("Feature window:", FEATURE_START, "to", FEATURE_END)
print("Outcome window:", OUTCOME_START, "to", OUTCOME_END)

Hugging Face READ token: ··········
Daily file downloaded successfully:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
Feature window: 2026-03-01 to 2026-03-31
Outcome window: 2026-04-01 to 2026-04-30


1. Unit of analysis + time window
One row in my analysis = one pseudonymized content item for the March 2026 decision window.

The underlying warehouse fact is daily: one row represents one report_date × client_hash_id × content_hash_id. I aggregate those daily rows into one March feature row per content item. The model/ranking decision is therefore made at the content-page level at the end of March.

Feature window: 2026-03-01 through 2026-03-31.

Outcome window: 2026-04-01 through 2026-04-30.

Decision moment: after March data is available and before April outcomes are known.

Label/proxy: is_declining_next_30d = 1 when April impressions are less than 80% of March impressions, i.e. more than a 20% decline. This is a future observed outcome, not a product decision flag.

## 2. Fields: feature / label / context / excluded

### Features — safe at the March decision moment
1. `march_impressions` — total GSC impressions during March.
2. `march_clicks` — total GSC clicks during March.
3. `march_avg_position` — average observed GSC position during March.
4. `march_days_with_impressions` — number of March days with at least one impression.
5. `content_age_days_at_decision` — content age measured at 2026-03-31.

### Label
- `is_declining_next_30d` — 1 when April impressions are below 80% of March impressions.

### Context
- `client_hash_id` — grouping/splitting only.
- `content_hash_id` — page identity and joins only.
- `report_date` — used to define windows, not a model feature.

### Excluded
- April impressions/clicks/position as model features — they are future information relative to the March decision.
- `trend_pct` / `trend_direction` from a current or overlapping window — they can contain the outcome period and therefore create leakage.
- `ga4_*` fields for this first contract — the lane can proceed using GSC signals, and early history can have `ga4_data_available = FALSE`.
- Product decision flags/scores such as refresh or priority flags — they would copy an existing business rule rather than discover signal.
- Raw/private client names, URLs, queries, and titles — not part of the released analysis.

## 3. Verify it with queries

The three query cells below are the required checks. They verify:

1. **Grain:** no duplicate daily fact rows for the stated source grain.
2. **Counts and window:** the March slice row count, distinct content items, and date span.
3. **Availability:** how many March rows remain when GSC availability is explicitly required with `IS TRUE`.

The notebook intentionally keeps this to **exactly three verification queries**.

In [66]:
# Grain check
q1 = f"""
SELECT
    COUNT(*) AS duplicate_grain_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
) d
"""

grain_check = con.sql(q1).df()

display(grain_check)

print("Expected result: duplicate_grain_groups = 0")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_grain_groups
0,0


Expected result: duplicate_grain_groups = 0


In [67]:
# VERIFICATION QUERY 2 — March count and date span
q2 = f"""
SELECT
    COUNT(*) AS march_fact_rows,
    COUNT(DISTINCT content_hash_id) AS march_content_items,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM {TABLES['daily']}
WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
"""

march_check = con.sql(q2).df()
display(march_check)

,march_fact_rows,march_content_items,min_report_date,max_report_date
0,9841378,331437,2026-03-01,2026-03-31


In [71]:
# VERIFICATION QUERY 3 — availability
# IS TRUE is deliberate: missing/unknown availability must not silently pass.
q3 = f"""
SELECT
    COUNT(*) AS march_rows_before_filter,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        * 100.0 / COUNT(*) AS gsc_available_pct
FROM {TABLES['daily']}
WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
"""

availability_check = con.sql(q3).df()
display(availability_check)


,march_rows_before_filter,gsc_available_rows,gsc_available_pct
0,9841378,3611061,36.692636


## 3b. Five-feature frame

I build the feature frame only from March data, so every feature is available at the decision moment. The April outcome is joined only afterward to create the label.

| Feature | Available when? |
|---|---|
| `march_impressions` | Known after the March reporting window closes because it is summed only from March GSC observations. |
| `march_clicks` | Known after March closes because it uses only March GSC click observations. |
| `march_avg_position` | Known after March closes because it uses only March observed positions. |
| `march_days_with_impressions` | Known after March closes because it counts only March dates with impressions. |
| `content_age_days_at_decision` | Known on 2026-03-31 because it is calculated from content creation date to the decision date. |

In [84]:
print(TABLES.keys())

dict_keys(['daily'])


In [83]:
print(TABLES['daily'])

read_parquet('/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet')


In [78]:
import os

snapshot_dir = "/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/"
for name in sorted(os.listdir(snapshot_dir)):
    print(name)

fact_content_daily_performance


In [79]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
top_level = sorted(set(f.split("/")[0] for f in files))
for t in top_level:
    print(t)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance
fact_content_daily_performance_sample.parquet
fact_content_query_90d.parquet


In [86]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print(len(hf_token))

37


In [87]:
con.sql(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
""")

TABLES['content'] = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

In [88]:
print(TABLES.keys())

dict_keys(['daily', 'content'])


In [90]:
display(con.sql(f"SELECT * FROM {TABLES['content']} LIMIT 5").df())

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [91]:
features_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_clicks,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS march_avg_position,
        COUNT(DISTINCT report_date)
            FILTER (WHERE COALESCE(gsc_impressions, 0) > 0) AS march_days_with_impressions
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1, 2
),
content_context AS (
    SELECT
        content_hash_id,
        MIN(content_created_date) AS content_created_date
    FROM {TABLES['content']}
    GROUP BY 1
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    m.march_days_with_impressions,
    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '{FEATURE_END}'
    ) AS content_age_days_at_decision
FROM march m
LEFT JOIN content_context c USING (content_hash_id)
WHERE m.march_impressions > 0
"""

features = con.sql(features_sql).df()
print(f"Feature rows: {len(features):,}")
display(features.head())
display(features.describe(include="all").T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 176,738


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_days_with_impressions,content_age_days_at_decision
0,client_73cda7b4e4f265ea,content_287262a30125d9e0,1475.0,2.0,3.165722,31,375
1,client_73cda7b4e4f265ea,content_653b0e6a2f03d727,433.0,0.0,17.085953,31,375
2,client_73cda7b4e4f265ea,content_bfd0d577371d456e,276.0,3.0,6.776587,29,375
3,client_73cda7b4e4f265ea,content_941dee84048d958a,159.0,0.0,9.024313,30,375
4,client_73cda7b4e4f265ea,content_f2c9500b9fed088a,481.0,2.0,8.047107,31,375


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
client_hash_id,176738,47,client_73cda7b4e4f265ea,27425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
content_hash_id,176738,176738,content_483b6f2bc9283cfc,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
march_impressions,176738.0,NaN,NaN,NaN,1587.986675,5431.337724,1.0,20.0,173.0,1039.0,617124.0
march_clicks,176738.0,NaN,NaN,NaN,4.650002,26.722649,0.0,0.0,0.0,2.0,5668.0
march_avg_position,175304.0,NaN,NaN,NaN,17.050555,18.333942,0.101639,5.5,9.0,22.0,309.0
march_days_with_impressions,176738.0,NaN,NaN,NaN,20.431718,11.480153,1.0,9.0,26.0,31.0,31.0
content_age_days_at_decision,176738.0,NaN,NaN,NaN,184.654545,123.634281,0.0,69.0,193.0,260.0,494.0


3c. Add the forward label — after the feature frame is built
The label is based on April impressions compared with March impressions. April data is not included in the five model features.

In [92]:
outcome_sql = f"""
WITH march AS (
    SELECT
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_impressions
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{FEATURE_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1
),
april AS (
    SELECT
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
    FROM {TABLES['daily']}
    WHERE report_date BETWEEN DATE '{OUTCOME_START}' AND DATE '{OUTCOME_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1
)
SELECT
    m.content_hash_id,
    m.march_impressions,
    COALESCE(a.april_impressions, 0) AS april_impressions,
    CASE
        WHEN COALESCE(a.april_impressions, 0) < 0.80 * m.march_impressions
        THEN 1 ELSE 0
    END AS is_declining_next_30d
FROM march m
LEFT JOIN april a USING (content_hash_id)
WHERE m.march_impressions > 0
"""

outcomes = con.sql(outcome_sql).df()

model_df = features.merge(
    outcomes[["content_hash_id", "april_impressions", "is_declining_next_30d"]],
    on="content_hash_id",
    how="inner"
)

print(f"Final feature/label rows: {len(model_df):,}")
print(f"Future-decline rate: {model_df['is_declining_next_30d'].mean():.3%}")
display(model_df.head())

Final feature/label rows: 176,738
Future-decline rate: 100.000%


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_days_with_impressions,content_age_days_at_decision,april_impressions,is_declining_next_30d
0,client_73cda7b4e4f265ea,content_287262a30125d9e0,1475.0,2.0,3.165722,31,375,0.0,1
1,client_73cda7b4e4f265ea,content_653b0e6a2f03d727,433.0,0.0,17.085953,31,375,0.0,1
2,client_73cda7b4e4f265ea,content_bfd0d577371d456e,276.0,3.0,6.776587,29,375,0.0,1
3,client_73cda7b4e4f265ea,content_941dee84048d958a,159.0,0.0,9.024313,30,375,0.0,1
4,client_73cda7b4e4f265ea,content_f2c9500b9fed088a,481.0,2.0,8.047107,31,375,0.0,1


## 3d. Deliberate leakage trap

To prove the leakage risk, I intentionally add **one label-derived column**: `leak_future_decline`, which is simply a copy of the future label.

This column is not a legitimate feature. It is added only to demonstrate how a model can appear nearly perfect when the answer is accidentally present in the inputs. After the demonstration, it is removed and the honest feature set is kept.

In [94]:
print(leak_df["is_declining_next_30d"].value_counts(dropna=False))

is_declining_next_30d
1    175304
Name: count, dtype: int64


In [98]:
print(model_df.columns.tolist())
print(model_df.dtypes)

['client_hash_id', 'content_hash_id', 'march_impressions', 'march_clicks', 'march_avg_position', 'march_days_with_impressions', 'content_age_days_at_decision', 'april_impressions', 'is_declining_next_30d']
client_hash_id                   object
content_hash_id                  object
march_impressions               float64
march_clicks                    float64
march_avg_position              float64
march_days_with_impressions       int64
content_age_days_at_decision      int64
april_impressions               float64
is_declining_next_30d             int32
dtype: object


In [99]:
import pandas as pd

# Look at the raw comparison across a sample of rows
sample = model_df[["march_impressions", "april_impressions", "is_declining_next_30d"]].sample(15, random_state=1)
display(sample)

# Check for missing/zero April data — this is the most common cause
print("april_impressions NaN count:", model_df["april_impressions"].isna().sum())
print("april_impressions == 0 count:", (model_df["april_impressions"] == 0).sum())
print("april_impressions describe:\n", model_df["april_impressions"].describe())

,march_impressions,april_impressions,is_declining_next_30d
166701,32.0,0.0,1
23063,1941.0,0.0,1
152848,144.0,0.0,1
19583,406.0,0.0,1
159156,17.0,0.0,1
119419,372.0,0.0,1
101379,4993.0,0.0,1
82464,1.0,0.0,1
103326,947.0,0.0,1
134048,54.0,0.0,1


april_impressions NaN count: 0
april_impressions == 0 count: 176738
april_impressions describe:
 count    176738.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
Name: april_impressions, dtype: float64


In [100]:
print("Rows where april_impressions >= march_impressions:", (model_df["april_impressions"] >= model_df["march_impressions"]).sum())
print("Rows where april_impressions is null:", model_df["april_impressions"].isna().sum())

Rows where april_impressions >= march_impressions: 0
Rows where april_impressions is null: 0


In [97]:
print(model_df["is_declining_next_30d"].value_counts(dropna=False))
print(model_df.shape)

is_declining_next_30d
1    176738
Name: count, dtype: int64
(176738, 9)


In [102]:
print(TABLES['daily'])

read_parquet('/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet')


In [103]:
TABLES['daily_april'] = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')"

# Sanity check — this should NOT be empty
check = con.sql(f"SELECT COUNT(*) AS n FROM {TABLES['daily_april']}").df()
print(check)

          n
0  10424730


In [104]:
april_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(COALESCE(gsc_impressions, 0)) AS april_impressions
FROM {TABLES['daily_april']}
WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
  AND gsc_data_available IS TRUE
GROUP BY 1, 2
"""

april_agg = con.sql(april_sql).df()
print(f"April rows: {len(april_agg):,}")
display(april_agg.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April rows: 194,760


,client_hash_id,content_hash_id,april_impressions
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0


In [105]:
model_df = features.merge(april_agg, on=["client_hash_id", "content_hash_id"], how="left")
model_df["april_impressions"] = model_df["april_impressions"].fillna(0)

model_df["is_declining_next_30d"] = (
    model_df["april_impressions"] < model_df["march_impressions"]
).astype(int)

print(model_df["is_declining_next_30d"].value_counts())

is_declining_next_30d
1    111968
0     64770
Name: count, dtype: int64


In [106]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

honest_features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_days_with_impressions",
    "content_age_days_at_decision",
]

leak_df = model_df.dropna(subset=honest_features).copy()

X_train, X_test, y_train, y_test = train_test_split(
    leak_df[honest_features],
    leak_df["is_declining_next_30d"],
    test_size=0.25,
    random_state=42,
    stratify=leak_df["is_declining_next_30d"],
)

honest_model = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
honest_model.fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])

# Deliberate leak: the future label itself is exposed as a feature.
leak_df["leak_future_decline"] = leak_df["is_declining_next_30d"]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    leak_df[honest_features + ["leak_future_decline"]],
    leak_df["is_declining_next_30d"],
    test_size=0.25,
    random_state=42,
    stratify=leak_df["is_declining_next_30d"],
)

leak_model = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
leak_model.fit(X_train_l, y_train_l)
leak_auc = roc_auc_score(y_test_l, leak_model.predict_proba(X_test_l)[:, 1])

print(f"Honest feature AUC: {honest_auc:.3f}")
print(f"Leaky feature AUC:  {leak_auc:.3f}")
print("Leak removed from final feature list:", "leak_future_decline" not in honest_features)

Honest feature AUC: 0.720
Leaky feature AUC:  1.000
Leak removed from final feature list: True


### Leakage lesson

The leaky score should move very close to perfect because the model has been handed the answer. That result is **not evidence of a good model**; it is evidence that the feature definition is invalid.

I therefore remove `leak_future_decline` and keep only the five March features. The honest score is the number I would report for this experiment.

The same rule applies to any current-window `trend_pct`, `trend_direction`, April metrics, or product decision field that overlaps the future outcome.

## 4. Data limits

1. **Unbalanced history:** clients do not all have the same tracking history, so March coverage is not equally deep across clients.
2. **GSC-only early rows:** GA4 availability can be false in early history. This contract therefore uses GSC signals and explicitly checks `gsc_data_available IS TRUE`.
3. **The label is directional:** a >20% month-over-month impression decline is a defined proxy for review risk. It does not prove that a page needs a refresh or that refreshing it will recover traffic.
4. **No causal claim:** this analysis cannot tell whether an update caused a performance change.
5. **Window dependence:** March features predict an April outcome. Features from April or overlapping 90-day windows would break the decision-time boundary and create leakage.
6. **Selection bias:** requiring measurable GSC availability and March impressions means the resulting ranking applies to observable content, not every piece of content in the warehouse.

## 5. Self-check

- [x] Unit of analysis and feature/outcome windows are stated.
- [x] Feature, label, context, and excluded fields are separated.
- [x] Exactly three verification query cells are included.
- [x] Availability is checked explicitly with `IS TRUE`.
- [x] Five features have an “available when?” explanation.
- [x] One deliberate label-derived leakage feature is added, tested, and removed.
- [x] Limitations are stated with careful, decision-support language.
- [ ] Run **Runtime → Run all** in Colab after adding your Hugging Face Secret and confirm the three query outputs and leakage scores are visible.
- [ ] Commit this executed notebook as `work/notebooks/w03_data_contract.ipynb`.